Groundwater | Flow Modeling Track

# Step 2: Perceptual Model – The Limmat Valley Aquifer

> **The 10 steps at a glance:** 1-Goal → **2-Perceptual** → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 1, you defined clear objectives for our Limmat Valley model: simulate steady-state flow, reproduce observed heads, and quantify river-aquifer exchange. Now we build a **perceptual model** – a detailed mental map of how the aquifer system works – before translating it into code.

In [ ]:
# Setup
import sys
import os

# Fix PROJ library path - must be set BEFORE importing geospatial libraries
# This prevents conflicts with other PROJ installations (e.g., anaconda)
import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import geopandas as gpd
import plotly.express as px

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file
import print_images as du
from map_utils import (
    display_groundwater_resources_map, 
    plot_model_area_map,
    display_concessions_map, 
    create_interactive_dem_map,
)
import climate_utils as cu
from climate_utils import get_and_plot_climate_data
import river_utils as ru
from river_utils import get_and_display_river_water_level_data, get_river_area
from perceptual_model_utils import get_and_plot_groundwater_levels

> **📘 Unit Convention**
>
> Field measurements and literature values often report hydraulic conductivity in **m/s**. For MODFLOW modeling, we convert to **m/day**:
>
> $K = 0.01 \text{ m/s} = 0.01 \times 86400 = 864 \text{ m/day}$
>
> Throughout this notebook, we use m/s for consistency with literature. In Notebook 4, all values will be converted to m/day for MODFLOW input.

| **Core content** | ~55-60 minutes |
|:--|:--|
| **Optional: Detailed calculations & data exploration** | +25-30 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Identify** the key hydrogeological features of the Limmat Valley aquifer
2. **Explain** the main water balance components (recharge, river interaction, pumping, outflow)
3. **Estimate** rough magnitudes of groundwater fluxes using simple calculations
4. **Construct** a perceptual model diagram summarizing system understanding

## Prerequisites

Before starting this notebook, you should have:
- **Completed [0_start_here.ipynb](../0_start_here.ipynb)** – explains the 10-step framework
- **Completed [1_model_goal.ipynb](1_model_goal.ipynb)** – defines what our model needs to achieve
- Basic understanding of Darcy's Law and aquifer properties
- Familiarity with water balance concepts

> **Where does data go?** Files downloaded in this notebook are saved to `~/applied_groundwater_modelling_data/`. The download functions print the exact path each time.

---

## Introduction

In this notebook, we build a **perceptual model** of the Limmat Valley aquifer – a mental map of how the groundwater system works before we translate it into a numerical model.

We follow the steps a professional groundwater modeler would take:
1. Understand the aquifer geometry and boundaries (Chapters 1-2)
2. Identify the main water fluxes: climate forcing (Chapter 3), rivers (Chapter 4), pumping (Chapter 5)
3. Review available monitoring data (Chapter 6)
4. Synthesize everything into a conceptual diagram (Summary)

> **Tip: Use the Outline Panel**
> This is a long notebook with many sections. To keep track of where you are:
> - **In VS Code**: Open the Outline panel (View → Open View → Outline)
> - **In JupyterHub/JupyterLab**: Open the Table of Contents panel (View → Table of Contents)

We start with a generic perceptual model (see figure below) and refine it as we gather information.

In [ ]:
du.display_image(
    image_filename='perceptual_model_00_initial.png', 
    image_folder='1_perceptual_model',
    caption='Initial perceptual model of the Limmat valley aquifer (indicated in brown) which will be refined and used as a basis for a first long-term average water balance estimation. The purple arrows indicate important groundwater fluxes. Qin represent lateral inflows, while Qout represents outflows. Qpump represents groundwater extraction. Qrech and Qleak stand for areal recharge and leakage from surface water bodies respectively. ∆S stands for the storage change in the aquifer.'
)

We first aim to understand the hydrogeology of the Limmat Valley Aquifer. Next, we examine each water balance component to develop an initial estimate of the key fluxes within the system. Finally, we will revisit and refine the perceptual model as we prepare to construct a numerical model of the Limmat Valley Aquifer.

## 1 - The Limmat Valley Aquifer

The Limmat Valley Aquifer is a well-studied groundwater reservoir beneath the city of Zurich, Switzerland. According to Doppler and colleagues, it was formed during the last ice age as the Linth glacier retreated. The aquifer has no direct hydraulic connection to Lake Zurich in the east, where it is bounded by impermeable lake sediments and moraine material. It is further confined in the north and south by the side moraines of the Linth glacier. Lateral inflow of groundwater from the surrounding hills is expected. The aquifer is also hydraulically connected to the river Sihl in the east and the river Limmat in the north. Its hydraulic properties are highly heterogeneous due to a complex geological history shaped by various sedimentation and erosion events from the rivers Sihl and Limmat [\[1, 2\]](#References).

In the figure below, you see a screenshot of the [GIS-browser](https://geo.zh.ch/maps?x=2677655&y=1253620&scale=396217&basemap=arelkbackgroundzh) of the canton of Zurich [\[3\]](#References). The cyan-blue area in the center of the map shows the Limmat Valley Aquifer. The darker the color, the greater the thickness of the aquifer. The GIS-browser is only available in German, but you can use your browser's translation feature to view it in your preferred language.

In [ ]:
du.display_image(
    image_filename='GIS-browser_canton_Zurich.png', 
    image_folder='1_perceptual_model',
    caption='Printscreen of the GIS-browser of the canton of Zurich displaying the cantonal groundwater map. Source: https://geo.zh.ch/, accessed 2024-05-01'
)

> **Action point:**  
> Take a few minutes of your time to browse the available data layers in the GIS-browser. There are many! Focus on the layers that are relevant to the Limmat valley aquifer.

We now begin systematically reviewing the available information to refine our perceptual model of the Limmat Valley Aquifer. The GIS-browser will help us clarify the aquifer's geometry and inform how we represent it in the numerical model.

You can use the checklist below to track your progress.

## 2 - Aquifer Geometry

Most layers in the GIS-Browser of the Canton of Zurich are publicly available for download (see the "Datenbezug" button in the upper right corner of the map pane). We use this feature to import the relevant data into our JupyterHub environment.

### 2.1 - Bottom Topography

We have downloaded the aquifer thickness layer from the GIS-browser and clipped it to the area of interest (AOI) for the Limmat Valley Aquifer. Let's examine the bottom topography of the aquifer in the figure below.


In [ ]:
# Download the groundwater map of the canton of Zurich
gw_map_path = download_named_file(
    name='groundwater_map_norm', 
    data_type='gis', 
)
print(f"✓ Data saved to: {gw_map_path}")

In [ ]:
# Create and display the interactive map
interactive_map = display_groundwater_resources_map(
    gw_map_path, zoom_level=11, 
    map_title="Groundwater resources map of the canton of Zurich",)
interactive_map

We can use contours of the aquifer thickness to build the bottom of the aquifer model (this is done in a future chapter : Notebook 4, model implementation).  

### 2.2 - Top Topography
The Limmat Valley Aquifer is unconfined (the water table is the upper boundary). We can use the land surface topography to represent the top of the aquifer. 

**SwissTopo** (the Swiss Federal Office of Topography) provides digital elevation models (**DEMs** – gridded representations of surface elevation) of Switzerland in various resolutions. We use the high-resolution swissALTI3D dataset with 2m resolution for our case study. This resolution is sufficient to capture important topographic features such as river beds, which is crucial for accurately modeling river-aquifer interactions.

If you are interested in learning more about the DEM and how to clip it to the area of interest (AOI), you can read the [processing_DEM.ipynb](../../_SUPPORT/src/scripts/scripts_limmat_data_preprocessing/processing_DEM.ipynb) notebook.

In [ ]:
# Download the high-resolution DEM
dem_path = download_named_file(
    name='dem_hres',
    data_type='gis'
)
print(f"✓ Data saved to: {dem_path}")

In [ ]:
m = create_interactive_dem_map(
    dem_path=dem_path, 
    zoom_start=12,
    container_title="DEM Controls",
    custom_title="High-resolution digital elevation model (swissALTI3D, 2m) in the area of interest. Background: OpenStreetMap"
)
m

The topography of the Limmat valley is quite flat, with a maximum elevation of about 400 m above sea level. With the 2m resolution of swissALTI3D, we can resolve important topographic features including river beds, which is essential for accurate river-aquifer interaction modeling.

River-aquifer interaction is predominantly driven by the stream bed elevation relative to the groundwater table. Therefore, an accurate representation of the streambed elevation in the numerical groundwater flow model is crucial. The high-resolution DEM allows us to better represent these features in our model. We will discuss the [river-aquifer interaction](#4---River-aquifer-interaction) in a later chapter.

### 2.3 - Lateral Model Boundaries
The aquifer is bounded by impermeable lake sediments and moraine in the east (no connection to Lake Zurich) and by side moraines in the north and south. **Lateral inflow** from the surrounding hills is estimated using a water balance approach (see [Chapter 3](#3---Climate-Forcing)).

For flow estimates using Darcy's law, we need **hydraulic conductivity (K)** – a measure of how easily water flows through the aquifer material, typically in m/s or m/day. K values are typically provided by the commissioning organization or estimated from literature (e.g., Freeze and Cherry, 1979, figure below).

In [ ]:
du.display_image(
    image_filename='hydraulic_conductivities_by_Freeze.png', 
    image_folder='1_perceptual_model',
    caption='Hydraulic conductivity ranges as presented in the book <<Groundwater>> by Freeze and Cherry (1979). The ranges are based on the hydraulic conductivity of unconsolidated sediments. Hydraulic conductivity is typically known only in approximate orders of magnitude.'
)

<details>
<summary><strong>Optional: Theory reminder – The Dupuit Approximation</strong></summary>


> To apply Darcy's Law for calculating the total discharge through an unconfined aquifer, we often rely on a set of simplifying assumptions known as the *Dupuit approximation* (or Dupuit-Forchheimer assumption). These assumptions allow us to treat the complex three-dimensional flow problem as a simpler two-dimensional one by integrating flow over the entire aquifer thickness.
> 
> The key conditions that must be met for the Dupuit approximation to be valid are:
> 
> *   *The water table gradient is small:* The slope of the phreatic surface must be gentle. This is the most critical assumption, as it implies that flow is almost entirely horizontal.
> *   *Flow is horizontal:* It is assumed that the hydraulic head is constant with depth for any given vertical line. This means equipotential lines are vertical, and therefore, the flow lines are horizontal.
> *   *The hydraulic gradient is equal to the slope of the water table:* The slope of the water table is assumed to be the hydraulic gradient (`i`) for throughout the saturated thickness of the aquifer.
> 
> When these conditions hold, we can use a modified form of Darcy's Law to calculate the discharge `Q` of the aquifer:
> 
> `Q = -K * h * B * (dh/dx)`
> 
> Where:
> *   `Q` is the discharge (e.g., m³/s)
> *   `K` is the hydraulic conductivity (e.g., m/s)
> *   `h` is the saturated thickness (m)
> *   `B` is the width of the aquifer perpendicular to the flow direction (m)
> *   `dh/dx` is the slope of the water table, which represents the hydraulic gradient.
> 
> In essence, the Dupuit approximation simplifies reality by ignoring vertical flow components, which is reasonable when the water table is nearly flat relatively to the saturated thickness.

</details>

For the outflow, a fixed-head boundary is typically chosen. This requires a known groundwater table at the boundary, e.g. a monitoring piezometer. Another option is to use the water level of a receiving stream or lake as a fixed-head boundary condition. The outflow boundary should be placed far enough away from the area of interest, so that the groundwater flow in the area of interest is not influenced by the boundary condition. For a regional model, this is typically several hundred meters to several kilometers away from the area of interest. 

In our case, we choose the groundwater level at piezometer 481 (indicated with a pink arrow in the figure below) as the fixed-head boundary condition. This relieves us from the need to implement the more complex aquifer geometry and river-aquifer interaction further downstream in the Limmat valley aquifer. We can use the isoline of equal groundwater level at 390 m a.s.l. from the GIS browser to define the outflow boundary.

In [ ]:
du.display_image(
    image_filename='Detail_outflow.png', 
    image_folder='1_perceptual_model',
    caption='Detail of the outflow boundary condition at piezometer 481 (purple arrow). The groundwater level at this piezometer is used as a fixed-head boundary condition for the groundwater flow model. The outflow boundary is placed far enough away from the area of interest, so that the groundwater flow in the area of interest is not influenced by the boundary condition.'
)

Note that the isolines of equal groundwater levels are not perfectly perpendicular to the flow direction but slightly curved at the aquifer boundaries (particularly visible at the northern boundary). This indicates lateral groundwater inflow from the north.  

We can now make a rough estimate of the groundwater flux at the western outflow boundary of the Limmat Valley Aquifer using the Dupuit's assumption. We use the groundwater levels isolines of 389 m a.s.l. and 391 m a.s.l. to estimate the hydraulic gradient. The distance between the two isolines can be measured in a GIS tool and is about 780 m. The thickness of the aquifer in that location is between 2 m and 10 m, and the width is approximately 1 km. We estimate the hydraulic conductivity to be around 10^-2 m/s so we can estimate the hydraulic gradient as follows:

> **📘 Note on Dupuit approximation accuracy:**
> We use the linear form of Darcy's Law ($Q = K \cdot i \cdot A$) rather than the more accurate $h^2$ form for unconfined flow. This introduces approximately 10–15% error when head changes are large relative to saturated thickness. In our case, $\Delta h / h_{avg} \approx 2/6 \approx 33\%$, so the linear approximation is acceptable for this rough estimate but introduces some error.

In [ ]:
# Estimate groundwater outflow at western boundary using Darcy's Law
# Q = K * i * A = K * (dh/dx) * thickness * width

# Hydraulic gradient from groundwater isolines
head_upstream = 391  # m a.s.l. (from GIS isoline)
head_downstream = 389  # m a.s.l. (from GIS isoline)
distance = 780  # m (measured between isolines)
gradient = (head_upstream - head_downstream) / distance  # m/m

# Aquifer geometry at outflow boundary
thickness_min = 2  # m (from GIS thickness map)
thickness_max = 10  # m (from GIS thickness map)
width = 1000  # m (aquifer width at boundary)

# Hydraulic conductivity (glaciofluvial gravels)
K_m_s = 0.01  # m/s (typical value for coarse gravels)
seconds_per_day = 86400

# Calculate outflow range
outflow_lower_bound = gradient * thickness_min * width * K_m_s * seconds_per_day  # m³/day
outflow_upper_bound = gradient * thickness_max * width * K_m_s * seconds_per_day  # m³/day

print(f"Hydraulic gradient: {gradient:.4f} m/m")
print(f"Estimated groundwater outflow at the western boundary:")
print(f"  Lower bound (thickness={thickness_min}m): {outflow_lower_bound:.0f} m³/day")
print(f"  Upper bound (thickness={thickness_max}m): {outflow_upper_bound:.0f} m³/day")

We now have an understanding of the geometry of the aquifer. Let's update our virtual perceptual model with the information we have gathered so far: 

- The aquifer does not have a direct hydraulic connection to lake Zurich in the east. 
- It is probably heavily influenced by the river Sihl in the east and the river Limmat in the north.
- Since groundwater flow in river valleys generally follows the topography, we can assume that the groundwater flow in the Limmat valley aquifer is directed from south-east to west. This general flow direction is corroborated by the groundwater flow arrows which become visible when we zoom in on the GIS-browser map (visit the GIS browser to zoom in).
- The aquifer thickness is highly variable. To adequately represent the known geometry of the aquifer, a 3D model with a high spatial resolution is required. We will first opt for a 2D model with a simplified geometry for the case study work to keep the model lean and fast on JupyterHub.
- The simplified geometry will show a larger thickness in the south-east with a gradual thinning towards the west. The thickness will be represented by a single layer with a variable thickness. We will use the thickness values from the GIS-browser to define the thickness of the aquifer in our model whereby we will have to map the isolines of groundwater thickness to the model grid.
- The aquifer extends beyond the boundaries of the city of Zurich. Since we are interested mainly in applying our numerical experiment in the city region, this means that we will have to make assumptions about the outflow boundary of the aquifer.

> **⚠️ Uncertainty:** Our Darcy-based outflow estimate depends linearly on hydraulic conductivity (K). While we assume K ≈ 10⁻² m/s, literature values for glaciofluvial gravels span an order of magnitude (10⁻³ to 10⁻¹ m/s). If K is constrained to within a factor of 2–3 through calibration, **expected uncertainty: ±50%**. Without calibration, the true uncertainty could be a factor of 10.

### 2.4 - Drawing the model boundary polygon
We can now draw the model boundary polygon manually in QGIS and export it as a geopackage layer. Given the complex aquifer thickness, it is the fastest method. This layer will define the model area in our numerical model.

If you need instructions of how to use QGIS to draw and edit a polygon, please refer to resources available online. 

In [ ]:
model_boundary_path = download_named_file(
    name='model_boundary',
    data_type='gis'
)

# Get the area of the model domain
gdf_boundary = gpd.read_file(model_boundary_path)
model_area_m2 = gdf_boundary.geometry.area.sum()
model_area_km2 = model_area_m2 / 1e6  # Convert to km²
print(f"Model area: {model_area_km2:.2f} km²")  

plot_model_area_map(
    gw_depth_path=gw_map_path, 
    model_boundary_path=model_boundary_path,
    custom_title="Model region with the model boundary in black. Base layer by OpenStreetMap.",
    basemap="osm", 
    basemap_alpha=0.9
)

## 3 - Climate Forcing

### 3.1 - Areal Recharge
We get climate data from the Swiss Federal Office of Meteorology and Climatology (MeteoSwiss). The data is available for viewing on [MeteoSwiss](https://www.meteoswiss.admin.ch/services-and-publications/applications/measurement-values.html#param=messnetz-klima&table=false&station=SMA&compare=y&chart=year) website [\[6\]](#References) and was downloaded via the [opendata.swiss plattform](https://opendata.swiss/en/dataset/klimanormwerte) [\[7\]](#References). It is available in the data directory of the case study repository. 
The closest station to the Limmat Valley Aquifer is Fluntern. We use data from there, which is available for the time period 1991-2020. The figure below shows the monthly average temperature and precipitation for the Fluntern station. 

In [ ]:
# Download, read, and plot climate data for the Fluntern station
climate_data_path, climate_norms = get_and_plot_climate_data(
    custom_title="Climate data for the Fluntern station."
)

> **🤔 Think about it**  
> We are looking at the climate data mostly to get an idea about groundwater recharge. In an aquifer located in a densely populated area, we have to consider that the recharge is not only influenced by the climate but also by human activities. What do you think are the most important human activities that influence groundwater recharge in this area?
>
> Hint: Consider how urbanization, land use changes, and water management practices can affect infiltration and runoff patterns.

Now we need to get from climate data to net recharge at the groundwater table, taking into account the mostly built-up land cover. You have several options to do this which are listed below by increasing complexity :  

- Use net recharge literature values in the project area, or reported fractions from precipitation in comparable areas ( ie. of similar climatic and land cover conditions)
- Estimate net recharge using a water balance approach (precipitation minus evaporation)
- Use a soil water balance model (e.g. Hydrus) or a land surface model  
- Simulate the water flow through the unsaturated zone in Modflow (requires high-resolution 3D grid for numerical stability)

We start out with the less complex option. Lerner states that between 5% and 15% of the precipitation infiltrates into the aquifer [\[8\]](#References). We will use a value of 10% as a first approximation, ie. 110 mm/year.
The rest of the precipitation evaporates (approximately 20% according to Qui et al. (2023)) or runs off into the Sihl or the Limmat (via the rainwater drainage system). 

Once the area of the model domain is known, we can calculate the total volume of areal recharge to the aquifer. 

> **⚠️ Uncertainty:** Applying a uniform recharge fraction across an urban area is a major simplification. Recharge varies due to impervious surfaces, leaking infrastructure, and artificial drainage. Literature values range from 5% to 30%. **Expected uncertainty: ±50%**

### 3.2 - Lateral Inflow from Hills

The lateral inflow from the hills is estimated using a water balance approach on the hill sides that contribute to the aquifer. 

Information needed are :
- climate data (as for the areal recharge in the previous paragraph)
- hills area, which is estimated using QGIS (see the figure below)

We assume that 10% of the precipitation in a hill area infiltrates and is transported, eventually becoming lateral inflow into the aquifer.

In [ ]:
du.display_image(
    image_filename='rough_first_manual_catchment_delineation_to_estimate_area_for_lateral_inflow.png', 
    image_folder='1_perceptual_model',
    caption='Manual catchment delineation to estimate area for lateral inflow. The area is delineated using the height model of the area (here DHM200 by Swisstopo). The area is used to estimate the lateral inflow from the hills in the north and south of the Limmat valley aquifer. The delineation is not precise and should be refined in a later step. The area relevant for the lateral inflow estimation is 15 km2 in the south and 11 km2 in the north of the Limmat valley aquifer.'
)

Please note, this manual delineation is sufficient for a first rough estimation. A proper catchment delineation would require a more detailed analysis of the topography and the hydrological conditions in the area. You find step-by-step instructions on how to delineate a catchment in many places on the internet, for example here: [https://hydrosolutions.github.io/caham_book/geospatial_data.html#sec-catchment-delineation](https://hydrosolutions.github.io/caham_book/geospatial_data.html#sec-catchment-delineation). 

> **⚠️ Uncertainty:** Our simplified catchment delineation and assumed recharge coefficient introduce uncertainty. Actual subsurface flow paths and recharge rates in hillslopes vary spatially and temporally. **Expected uncertainty: ±40%**

In [ ]:
inflow_south = 0.11 * 15e6  # m3/year
inflow_north = 0.11 * 11e6  # m3/year

print(f"Estimated lateral inflow from the south: {(inflow_south/1000000):.1f} 10^6 m³/year")
print(f"Estimated lateral inflow from the north: {(inflow_north/1000000):.1f} 10^6 m³/year")

> **Think about it:**. 
> What type of boundary condition would you use to represent the lateral inflow from the hills in a steady state numerical model? Why?  
> 
> Hint: Remember, we know 3 types of boundary conditions: no-flow, fixed-head, and specified-flux.

## 4 - River-aquifer interaction
In most unconsolidated rock aquifers in river valleys, river-aquifer interaction is a key process that influences groundwater levels and flow patterns. The interaction between the river and the aquifer can lead to changes in water levels in the aquifer, which can affect the availability of groundwater resources. We therefore look at this process in more detail:

1. Understand the river geometry and its interaction with the aquifer (chapter [4.1](#41---introduction-to-river-geometry--recharge)).
2. Analyze monitoring data for river discharge and river water levels (chapter [4.2](#42---understanding-monitoring-data-river-discharge--river-water-levels)).
3. Estimate the leakage flux between the river and the aquifer (chapter [4.3](#43---estimating-the-leakage-flux)).

### 4.1 - Introduction to River Geometry & Recharge
The river-aquifer interaction is an important component of the groundwater flow model. The geometry of the river and its interaction with the aquifer influence both the direction of groundwater flow and the groundwater levels in the aquifer.

In practice, a modeler uses measured profiles of the riverbed to represent the river geometry in the model. For this course, we do not have access to riverbed profiles so we must make some assumptions about the river geometry.

> **💡 Tip: Visit your model area**
> If you are based in Zurich, you can visit the rivers. Visiting your model area is a great way to get a feel for the site and understand the hydrological processes at play. 

#### Optional 4.1.1 Interactive Exploration of River-Aquifer Interaction

The figure below provides an interactive tool to help you visualize the complex relationship between a river and an adjacent unconfined aquifer. It consists of two main parts:

1.  **Left Plot (Physical Cross-Section):** This shows a physical model of the river and the groundwater system. You can see the water level in the river (`H_riv`) and the water table in the aquifer (`H_aq`).
2.  **Right Plot (Flux Relationship):** This conceptual graph shows the calculated flux (flow rate, Q_riv) between the river and the aquifer as a function of the aquifer head. The red dot indicates the current state of the system shown on the left.

**How to Use This Figure:**

Use the two sliders below the plot to change the **Aquifer Head (`H_aq`)** and the **River Stage (`H_riv`)**. As you move the sliders, observe how the system changes in both plots. Pay close attention to the title, the equation, the direction of the flow (indicated by the arrow), and the position of the red dot on the flux curve.

**Scenarios to Explore:**

Try to create the following three fundamental conditions:

1.  **Gaining Stream:**
    *   **How to create it:** Set the aquifer head (`H_aq`) to be *higher* than the river stage (`H_riv`).
    *   **What to observe:** Water flows from the aquifer into the river. The flux (`Q_riv`) is negative (flow *out of* the groundwater). The head difference (`ΔH`) is between `H_aq` and `H_riv`. Nb : on the right plot, the red dot is in the negative flux region.

2.  **Losing Stream (Connected):**
    *   **How to create it:** Set the river stage (`H_riv`) to be *higher* than the aquifer head (`H_aq`), but keep `H_aq` *above* the riverbed bottom (`R_bot`, the dashed brown line).
    *   **What to observe:** Water flows from the river into the aquifer. The flux is positive and its magnitude depends directly on the head difference (`ΔH`). As you lower `H_aq`, the flux increases. The red dot on the right plot moves down the sloped part of the curve.

3.  **Losing Stream (Disconnected):**
    *   **How to create it:** Lower the aquifer head (`H_aq`) until it is *below* the riverbed bottom (`R_bot`).
    *   **What to observe:** A yellow "Vadose Zone" appears between the riverbed and the water table, indicating they are no longer directly connected. The flow from the river now seeps downward at a constant, maximum rate. The flux no longer depends on `H_aq`. Notice that the head difference (`ΔH`) for the calculation is now between `H_riv` and `R_bot`. On the right plot, the red dot moves onto the vertical part of the curve, showing that the flux remains constant even if you continue to lower `H_aq`.

### Comprehension Check

> **In your own words, what determines whether a stream is gaining or losing?**

<details>
<summary>Click to check your answer</summary>

A stream is **gaining** when the groundwater table is higher than the river stage—water flows from the aquifer into the river. A stream is **losing** when the river stage is higher than the groundwater table—water infiltrates from the river into the aquifer.

The key factors are:
1. **Head difference** between river and aquifer
2. **Riverbed conductance** (controlled by the clogging layer)
3. **Connection status** (connected vs. disconnected)

For disconnected streams, the flux becomes constant and depends only on river stage minus riverbed bottom elevation.
</details>

In [ ]:
ru.plot_river_aquifer_interaction(
    custom_title="Illustration of river-aquifer interaction and the corresponding flux vs. head relationship."
)

### 4.2 - Understanding Monitoring Data: River Discharge & River Water Levels

#### Optional 4.2.1 - Where to get the data from
River water level data is available from [BAFU](../glossary.md#swiss-data-sources) hydrological gauging stations. Relevant stations for our study area are **Sihl - Zürich, Sihlhölzli** (ID 2176) and **Limmat - Zürich, Unterhard** (ID 2099), shown in the figure below [\[10\]](#References).

In [ ]:
du.display_image(
    image_filename='BAFU_HydrologicalGaugingStations_closeup.png', 
    image_folder='1_perceptual_model',
    caption='Locations of hydrological gauging stations near the Limmat valley aquifer.'
)

> **🤔 Think about it**  
> For surface water balancing, discharge is the important variable. However, when it comes to flooding or river-aquifer interaction, the water level is the more important variable. Why?
>
> Hint: Consider how water levels relate to hydraulic gradients and flow directions between rivers and aquifers.

#### 4.2.2 - Summary of Water Level Data
Let's examine the water level data for the period 2010-2020.

In [ ]:
# Download, plot, and summarize river water level data
river_data_path, summary = get_and_display_river_water_level_data(
    start_year=2010,
    end_year=2020,
    figure_number=13
)

### 4.3 - Estimating the Leakage Flux
We need to compare river water levels and groundwater levels, to determine if the streams are gaining or losing water relatively to the aquifer and to estimate the water flux between the rivers and the aquifer. We use groundwater levels extracted form the mean and max isohypse layers in the GIS-browser at the locations of the river gauging stations. Based on this information, we can draw a conceptual figure of the river cross-sections with the river-aquifer interaction at the two gauging stations (see figures below).

In [ ]:
# Sihl River Data
sihl_gw_mean = 400.8
sihl_gw_high = 403.5
sihl_river_mean = 412.347
sihl_river_high = 413.439
sihl_depth_mean = 0.4

# Limmat River Data
limmat_gw_mean = 399.5
limmat_gw_high = 400.5
limmat_river_mean = 400.285
limmat_river_high = 401.822
limmat_depth_mean = 1.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14))
fig.suptitle("River and Groundwater Level Cross-Sections", fontsize=16, y=0.95)

# Plot Sihl
ru.plot_cross_section(ax1, "a) River Sihl", sihl_gw_mean, sihl_gw_high, sihl_river_mean, sihl_river_high, sihl_depth_mean)

# Plot Limmat
ru.plot_cross_section(ax2, "b) River Limmat", limmat_gw_mean, limmat_gw_high, limmat_river_mean, limmat_river_high, limmat_depth_mean)

plt.tight_layout(rect=[0, 0, 1, 0.93]) # Adjust layout to make space for suptitle
plt.show()

From the two conceptualized cross-sections at the gauge location on the river Sihl we see that we are dealing with a losing stream which is disconnected from the groundwater table. We therefore have a constant flux from the river into the aquifer. The flux is not influenced by the groundwater table in the aquifer. The flux is influenced only by the river stage and the river bed elevation.
The river Limmat is also a loosing stream at the location of the gauge. However, from the conceptualized cross-section we see that the river Limmat is very likely connected to the groundwater table. We have to account for a capillary fringe of up to 30 cm (depending on the pore size distribution of the soil matrix). The flux from the river into the aquifer is influenced by the groundwater table in the aquifer. The flux is further influenced by the river stage and the river bed elevation.

The river-aquifer interaction is limited by a **clogging layer** (German: *Kolmationsschicht* or *Schmutzschicht*), a thin layer of fine sediments that accumulates on riverbeds over time. This layer forms through:
- Deposition of fine suspended particles during low-flow periods
- Biological activity (biofilms, algal mats)
- Chemical precipitation (iron/manganese oxides)

The clogging layer is typically only a few centimeters thick but has a **hydraulic conductivity 100–10,000× lower** than the underlying aquifer material. It acts as a "bottleneck" controlling infiltration rates—even highly permeable gravel aquifers have limited exchange with rivers due to this layer. The leakage coefficient combines the clogging layer properties:

$$q_{riv} = \frac{K_{clogging}}{d_{clogging}} \cdot (H_{riv} - H_{aq}) = l_{leakage} \cdot (H_{riv} - H_{aq})$$

Where:
- $q_{riv}$ is the specific flux from the river into the aquifer ($m$ $s^{-1}$)
- $K_{clogging}$ is the hydraulic conductivity of the clogging layer ($m$ $s^{-1}$)
- $d_{clogging}$ is the thickness of the clogging layer ($m$) 
- $l_{leakage}$ is the leakage coefficient ($s^{-1}$)
- $H_{riv}$ is the river water level ($m$)
- $H_{aq}$ is the groundwater level in the aquifer ($m$)

> **📘 Leakage coefficient vs. MODFLOW conductance:**
> The specific leakage coefficient $l_{leakage}$ (units: 1/time) describes flux per unit head difference per unit area. In MODFLOW, the RIV package uses **conductance** $C$ (units: m²/day), which is: $C = l_{leakage} \times A_{riverbed}$, where $A_{riverbed}$ is the area of the riverbed in each model cell. This allows MODFLOW to calculate volumetric flux directly: $Q = C \cdot (H_{riv} - H_{aq})$.

Doppler et al. [\[12\]](#References) calibrated leakage coefficients for the Limmat Valley:
- **River Sihl**: $1.3 \times 10^{-6}$ $s^{-1}$ (relatively uniform)
- **River Limmat**: $3.5 \times 10^{-7}$ to $3.5 \times 10^{-5}$ $s^{-1}$ (two orders of magnitude range!)

### Estimating Infiltration: Why We Can Only Quantify the Sihl

For the **River Sihl**, we can make a reasonable estimate of infiltration:
- The river is **clearly disconnected** from the groundwater table (11.5 m head difference)
- The leakage coefficient is **relatively uniform** along the river
- The flux is constant and independent of groundwater levels

For the **River Limmat**, however, reliable estimation with rough methods is **not possible**. Here's why:

1. **Extreme parameter uncertainty**: The leakage coefficient spans **two orders of magnitude** (100× range) along the river due to:
   - Riverbed composition varying from gravel bars to fine sediment pools
   - Flow velocity effects (scouring vs. clogging accumulation)
   - Biological activity varying with light, temperature, and nutrients
   - Flood history impacts on clogging layer thickness

2. **Complex connectivity**: The Limmat is **connected to the groundwater table** in stretches, meaning:
   - Flux depends on both river stage AND aquifer head
   - The connectivity status changes spatially (and possibly seasonally)
   - Some downstream reaches may even have groundwater exfiltrating into the river

3. **Hardhof dominates**: The engineered Hardhof system infiltrates ~7.7 million m³/year of river filtrate—this concentrated, controlled infiltration **dwarfs and overlaps** with any diffuse natural exchange we might estimate

> **⚠️ Key insight: This is why we need a groundwater model**
> 
> The Limmat-aquifer exchange cannot be reliably quantified using the rough methods available at the perceptual model stage. A calibrated numerical model can constrain this uncertain parameter by:
> - Solving the coupled flow equations between river and aquifer
> - Adjusting the leakage coefficient until simulated groundwater levels match observations
> - Quantifying the posterior uncertainty after calibration
>
> This limitation is not a failure of our analysis—it's a **fundamental characteristic** of connected river-aquifer systems with heterogeneous riverbeds.

We therefore estimate infiltration **only for the Sihl** and acknowledge that the Limmat contribution will be determined through model calibration.

In [ ]:
# Estimate specific flux for Sihl only (disconnected losing stream)
# Note: For a truly disconnected stream, the driving head should be (H_riv - R_bot)
# rather than (H_riv - H_aq), since the unsaturated zone below the riverbed 
# is at atmospheric pressure. However, the leakage coefficients from Doppler et al.
# were calibrated using (H_riv - H_aq), so we use that formulation for consistency.

q_riv_sihl = 1.3e-6 * (sihl_river_mean - sihl_gw_mean)  # m/s

print(f"Estimated specific flux from Sihl to aquifer: {q_riv_sihl:.2e} m/s")
print(f"\nNote: Limmat flux not estimated due to 2-order-of-magnitude uncertainty in leakage coefficient")
print(f"      and complex connectivity. Will be determined through model calibration.")

The area of the river Sihl within our model boundary can be estimated from the geopackage layer "Oeffentliche Oberflaechengewaesser" provided by the Department of Geoinformation of the Canton of Zurich ([https://opendata.swiss/en/dataset/offentliche-oberflachengewasser](https://opendata.swiss/en/dataset/offentliche-oberflachengewasser), accessed 2025-07-06) using QGIS. 

> **⚠️ Uncertainty for Sihl estimate:** Even for the relatively simple Sihl system, uncertainty remains due to spatial variation in riverbed properties and our simplified geometry assumptions. **Expected uncertainty: factor of 3** (i.e., true flux could be 3× higher or lower than our estimate). This is still much better constrained than the Limmat, where uncertainty spans a factor of 10–100.

In [ ]:
# Get river areas within model boundary
river_areas = get_river_area(gw_map_path=gw_map_path, show_figures=False)
sihl_area = river_areas['sihl_area']
limmat_area = river_areas['limmat_area']

In [ ]:
# Calculate annual infiltration from Sihl only
river_sihl_inflow = sihl_area * q_riv_sihl * 365 * 24 * 3600  # m³/year

print(f"Estimated groundwater recharge from Sihl: {(river_sihl_inflow/1000000):.1f} 10^6 m³/year")
print(f"\nLimmat natural infiltration: Not estimated (requires model calibration)")

## 5 - Groundwater pumping

The Limmat Valley aquifer is heavily used for groundwater abstraction—for drinking water, industrial cooling, and heat pumps. Since actual pumping rates are not publicly available, we estimate abstraction from **concession data** (maximum permitted rates) in the cantonal GIS-browser.

**Key findings** (details in optional sections below):
- **Hardhof well field** dominates extraction: ~7 million m³/year for Zurich's drinking water supply
- **Thermal use** (heat pumps) is widespread but **non-consumptive**—water is extracted and reinjected
- Actual pumping can be lower than concessioned amounts (e.g., Hardhof uses only ~13% of its permitted volume)

### 5.1 - Active concessions in the model area

In [ ]:
# Download and load wells data (Grundwasserfassungen)
wells_path = download_named_file(name='wells', data_type='gis')
wells_gdf = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')

# Clip to model boundary and add concession ID
wells_gdf = wells_gdf.clip(gdf_boundary)
wells_gdf['concession_id'] = wells_gdf['GWR_ID'].str.split('_').str[0]

# Filter for active wells only (remove decommissioned and unused)
is_active = ~(
    wells_gdf['BESCHREIBUNG'].str.contains('aufgehoben', case=False, na=False) |
    wells_gdf['BESCHREIBUNG'].str.contains('ungenutzt', case=False, na=False)
)
wells_active = wells_gdf[is_active].copy()

# Summary
print(f"Active concessions in model area: {wells_active['concession_id'].nunique()}")
print(f"\nBreakdown by size category:")
print(wells_active.groupby('BESCHREIBUNG')['concession_id'].nunique().to_string())

The dataset uses German abbreviations for well usage types: **TW** = drinking water, **BW** = process water, **KW** = cooling water, **WPG** = groundwater heat pump. Wells sharing the same concession ID belong to the same water right.

Most concessions are for **thermal use** (heat pumps), which is non-consumptive: water is extracted, used for heating/cooling, and reinjected nearby. For our water balance, we focus on **consumptive uses** (drinking water, industrial water) in the large concession category (>3000 L/min).

We observe intensive thermal use of groundwater in the Limmat Valley aquifer. Thermal use is typically **non-consumptive**: water is abstracted in one well and returned to the aquifer in another after use.

For our water balance, we focus on the **Hardhof well field** (concession b1-71), which dominates groundwater extraction in the model area.

**Hardhof well field** is one of four pillars of the drinking water supply of the City of Zurich [\[13\]](#references) and contributes an average of 15% of the annual supply (most comes from treated lake water). The well field operates as a **managed aquifer recharge (MAR) system**: riverbank filtrate is extracted along the River Limmat, **artificially infiltrated** into 3 recharge basins and 12 infiltration wells, and finally collected by horizontal wells for distribution. According to the statistical yearbook of the City of Zurich, the water works produce around **7 million m³ of groundwater per year** [\[14\]](#references).

**Understanding Hardhof in the water balance:**

The Hardhof system creates two distinct fluxes across our aquifer boundary:
- **Pumping (outflow)**: ~7 million m³/year extracted and delivered as drinking water—this water **leaves** the aquifer system
- **Artificial recharge (inflow)**: ~7.7 million m³/year of river filtrate infiltrated into the aquifer—this water **enters** the aquifer

The source water is 100% riverbank filtrate pumped from the Limmat and infiltrated through recharge basins. This is **different from the diffuse natural infiltration** we estimated in Section 4, which occurs along the entire riverbed. Hardhof represents *concentrated, engineered* infiltration at specific locations.

**For our numerical model**, we will need to represent both processes:
- Natural river-aquifer exchange → RIV package (leakage through riverbed)
- Hardhof operations → WEL package (concentrated pumping and injection wells)

The concession allows 104,000 L/min, but only about 13% is actually extracted—illustrating that concession data represents maximum permitted volumes, not actual use.

> **⚠️ Uncertainty:** Hardhof operations are well-documented, but we assume all recharge water is 100% river filtrate. In reality, some ambient groundwater may mix in. **Expected uncertainty: ±20%**

> **Suggestion:** If you are in Zurich, we encourage you to visit the Hardhof well field on one of their free public guided tours.

### Optional: 5.2 - Detailed analysis of medium concessions

The following cells analyze medium-sized concessions (300-3000 L/min) to ensure we don't miss significant fluxes.

In [ ]:
id_col    = "concession_id"   # adjust if different
desc_col  = "BESCHREIBUNG"
use_col   = "NUTZART"

phrase_medium      = "Grundwasserfassung mit Ertrag 300 - 3000 l/min"
phrase_recharge  = "Grundwasseranreicherungsanlage, Rückversickerung, Sickergalerie"

# 1. Keep only wells with a non-empty NUTZART (still “active” set assumed already)
wells_valid = wells_active[
    wells_active[use_col].notna() &
    wells_active[use_col].astype(str).str.strip().ne("")
].copy()

# 2. Identify medium-yield wells
mask_medium = wells_valid[desc_col].str.contains(phrase_medium, case=False, na=False)
medium_yield_wells = wells_valid[mask_medium]

# 3. Concessions having at least one medium-yield well
concessions_with_medium = (
    medium_yield_wells[id_col]
    .dropna()
    .astype(str)
    .unique()
)

# 4. Subset ALL wells belonging to those concessions (this is the expanded set)
medium_concessions = wells_valid[
    wells_valid[id_col].astype(str).isin(concessions_with_medium)
].copy()

# 5. Classify well role inside those concessions
medium_concessions["WELL_GROUP"] = np.select(
    [
        medium_concessions[desc_col].str.contains(phrase_medium, case=False, na=False),
        medium_concessions[desc_col].str.contains(phrase_recharge, case=False, na=False),
    ],
    ["MediumYield", "Recharge"],
    default="OtherWithinConcession"
)

# (Optional) If you want to keep only wells that are either HighYield or Recharge
# plus all other wells in those *same* concessions (already done above).
# If you instead want to drop the 'OtherWithinConcession' group, uncomment:
# medium_concessions = medium_concessions[medium_concessions['WELL_GROUP'] != 'OtherWithinConcession'].copy()

print(f"Concessions with medium-yield wells: {len(concessions_with_medium)}")
print("Counts by WELL_GROUP:")
print(medium_concessions["WELL_GROUP"].value_counts())

# Count the number of concessions for TW and BW
print("Number of concessions for TW and BW:")
print(medium_concessions[medium_concessions[use_col].isin(["TW", "BW", "TW, BW"])][id_col].nunique())

display_concessions_map(
    medium_concessions, 
    boundary_gdf=gdf_boundary, 
    map_title="Wells belonging to medium concessions (300 - 3000 l/min) in the model area colored by concession ID and marked by use type. Small and large active concessions are not shown in this map."
)


In [ ]:
# List the active wells in medium concessions which are used for drinking water and industrial water use
active_wells_medium_concessions = medium_concessions[medium_concessions[use_col].isin(["TW", "BW", "TW, BW"])]
print(active_wells_medium_concessions[['GWR_ID', 'FASSBEZ', 'FASSART', 'NUTZART']])

By clicking on a well in the GIS-Browser, we can access slightly more information than what is provided in the geodata layer available here. The Lochergut concession (b010063) is for 520 L/min, and the Schlachthof concession is for 370 L/min. The concessioned pumping rate for concession b010113 is 2000 L/min. The two drinking water wells north of the Limmat (concessions n010039_01 and n010085_01) allow pumping of 1200 L/min and 3000 L/min, respectively.

Since the two drinking water wells north of the river are most likely pumping riverbank filtrate and are located far downstream of our focus area, we will not implement them for now. As we observed at Hardhof, only a fraction of the concessioned amount is typically extracted. For the purpose of this model, we will assume 40%. Under this assumption, the medium-sized concessions south of the Limmat would amount to about 600,000 m³/year (less than 10% of the Hardhof abstractions).

In [ ]:
# Hardhof well field: Managed Aquifer Recharge (MAR) system
# Two distinct fluxes: pumping (outflow) and artificial recharge (inflow)

hardhof_pumping = 7.0e6  # m³/year - extracted as drinking water (OUTFLOW)
hardhof_recharge = 7.7e6  # m³/year - river filtrate infiltrated (INFLOW)

# Net effect on aquifer storage
hardhof_net = hardhof_recharge - hardhof_pumping  # m³/year (positive = net inflow)

print(f"Hardhof pumping (outflow):     {hardhof_pumping/1e6:.1f} 10^6 m³/year")
print(f"Hardhof recharge (inflow):     {hardhof_recharge/1e6:.1f} 10^6 m³/year")
print(f"Hardhof net effect:            {hardhof_net/1e6:+.1f} 10^6 m³/year")
print(f"\nNote: All Hardhof water is river filtrate from the Limmat.")

## 6 - Monitoring the Limmat Valley Aquifer

To gain a first impression of the groundwater levels in the Limmat valley aquifer, we will have a look at the groundwater map.

Several authorities do groundwater monitoring in the Limmat valley aquifer. We will start with the federal office for the environment (FOEN) which maintains a network of groundwater observation wells. You find an overview over the available groundwater monitoring sites at [https://map.geo.admin.ch/](https://map.geo.admin.ch/) in layer *Groundwater level/spring discharge* [\[15\]](#References). One monitoring well maintained by FOEN is located in the Limmat valley aquifer but far downstream of our area of interest in the city center. Further, this well is a drinking water production well and can therefore not be used as an outflow boundary.  

Few groundwater observation wells are operated by the cantonal office of the environment ([AWEL](../glossary.md#swiss-data-sources)) [\[16\]](#References) (see figure below).

In [ ]:
du.display_image(
    image_filename='GWmonitoring_locations_AWEL.png', 
    image_folder='1_perceptual_model',
    caption='Monitoring wells in the Limmat valley aquifer. Yearbook sheets for each site can be accessed through the popup window from each site. Source: https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/grundwasserstaende.html, accessed 2025-05-01.'
)

### 6.1 - Groundwater Level Trends

To verify that the aquifer is in steady state, we analyze groundwater level data from AWEL cantonal monitoring wells. If the system is in steady state, we expect no significant long-term trends.

In [ ]:
# Load and plot groundwater level trends from AWEL monitoring wells
gw_ts_path, gw_annual = get_and_plot_groundwater_levels()

### 6.2 - Steady-State Assumption

Looking at the annual mean groundwater levels over the past decades, no significant long-term trends are visible. We can assume that the groundwater levels have remained relatively stable over this period and that the system is in **steady state**.

$$ \Delta S = 0 $$

This justifies using a steady-state model for our initial groundwater flow simulation.

## Water Balance

For a control volume enclosing the aquifer, conservation of mass requires:

$$\Delta S = \sum Q_{in} - \sum Q_{out}$$

Where:
- $\Delta S$ = change in aquifer storage (m³/year)
- $\sum Q_{in}$ = sum of all inflows (m³/year)
- $\sum Q_{out}$ = sum of all outflows (m³/year)

Based on our analysis of groundwater level trends (Section 6), the system is in **steady state**, so $\Delta S = 0$ and:

$$\sum Q_{in} = \sum Q_{out}$$

The fluxes we identified in this notebook are:

| **Inflows** | **Outflows** |
|:--|:--|
| Areal recharge ($Q_{rech}$) | Groundwater outflow west ($Q_{out}$) |
| Lateral inflow from hills ($Q_{in}$) | Pumping ($Q_{pump}$) |
| River leakage from Sihl ($Q_{leak,Sihl}$) | |
| River leakage from Limmat ($Q_{leak,Limmat}$) | |
| Hardhof artificial recharge ($Q_{Hardhof,in}$) | |

The figure below shows these fluxes in our refined perceptual model, followed by our quantitative estimates.

### Comprehension Check

> **Which flux in the water balance has the highest uncertainty, and why?**

<details>
<summary>Click to check your answer</summary>

**River Limmat natural infiltration** has the highest uncertainty (×10–100, or 1–2 orders of magnitude).

This extreme uncertainty arises from:
1. The leakage coefficient varies spatially by a factor of 100 along the river
2. The river is partially connected to the groundwater table, making flux dependent on both river and aquifer heads
3. The engineered Hardhof system overlaps with and dominates natural infiltration

This is fundamentally different from other fluxes like areal recharge (±50%) or Sihl infiltration (×3), where we can at least make reasonable point estimates. The Limmat flux **cannot be reliably estimated** without a calibrated numerical model.
</details>

In [ ]:
# Display figure with updated fluxes
# TODO: FIGURE NEEDS UPDATE WITH FINAL FLUXES
du.display_image(
    image_filename='perceptual_model_01_final.png', 
    image_folder='1_perceptual_model', 
    caption='Final perceptual model of the Limmat valley aquifer with estimated fluxes. The purple arrows indicate important groundwater fluxes. Qin represent lateral inflows, while Qout represents outflows. Qpump represents groundwater extraction. Qrech and Qleak stand for areal recharge and leakage from surface water bodies respectively. ∆S stands for the storage change in the aquifer.'
)

In [ ]:
# Calculate the fluxes for the water balance
# Using 10% of precipitation (1100 mm/year) = 110 mm/year net recharge
net_recharge = 0.11 * model_area_km2 * 1000000  # m³/year (110 mm/year = 0.11 m/year)
river_sihl_inflow = sihl_area * q_riv_sihl * 365 * 24 * 3600  # m³/year
gw_outflow_west = (outflow_lower_bound + outflow_upper_bound) / 2 * 365  # m³/year

# Water balance summary with uncertainty ranges
# Limmat natural infiltration is NOT included - requires model calibration
water_balance = pd.DataFrame({
    "Component": [
        "Net areal recharge",
        "Lateral inflow (south)",
        "Lateral inflow (north)",
        "River Sihl (natural infiltration)",
        "River Limmat (natural infiltration)",
        "Hardhof recharge (river filtrate)",
        "Hardhof pumping (drinking water)",
        "GW outflow (west)",
    ],
    "Type": ["In", "In", "In", "In", "In", "In", "Out", "Out"],
    "Estimate (10⁶ m³/yr)": [
        round(net_recharge / 1e6, 1),
        round(inflow_south / 1e6, 1),
        round(inflow_north / 1e6, 1),
        round(river_sihl_inflow / 1e6, 1),
        "?",  # Limmat - cannot be estimated with rough methods
        round(hardhof_recharge / 1e6, 1),
        round(hardhof_pumping / 1e6, 1),
        round(gw_outflow_west / 1e6, 1),
    ],
    "Uncertainty": ["±50%", "±40%", "±40%", "×3", "×10–100", "±20%", "±20%", "±50%"],
    "Notes": [
        "",
        "",
        "",
        "Disconnected, uniform",
        "Requires calibration",
        "100% river filtrate",
        "",
        "",
    ],
})

print("Water Balance Summary")
print("=" * 90)
print(water_balance.to_string(index=False))
print("")

# Calculate totals excluding the unknown Limmat contribution
known_inflows = water_balance[
    (water_balance["Type"] == "In") & 
    (water_balance["Estimate (10⁶ m³/yr)"] != "?")
]["Estimate (10⁶ m³/yr)"].astype(float).sum()
total_out = water_balance[water_balance["Type"] == "Out"]["Estimate (10⁶ m³/yr)"].astype(float).sum()

print(f"Known inflows:     {known_inflows:.1f} 10⁶ m³/year")
print(f"Total outflows:    {total_out:.1f} 10⁶ m³/year")
print(f"Apparent gap:      {known_inflows - total_out:+.1f} 10⁶ m³/year")
print("")
print("Note: The Limmat natural infiltration (marked '?') cannot be reliably estimated")
print("      with rough methods due to 2-order-of-magnitude uncertainty in leakage")
print("      coefficients and complex river-aquifer connectivity.")
print("")
print("      This unknown flux will be constrained through model calibration against")
print("      observed groundwater levels.")

### Interpreting the Water Balance

The water balance table above reveals a key limitation of perceptual modeling: **we cannot close the budget** because one of the dominant fluxes—Limmat-aquifer exchange—cannot be reliably estimated.

**What we can quantify:**
- **Areal recharge** and **lateral inflows** from precipitation (with ±40–50% uncertainty)
- **Sihl infiltration**: The disconnected, uniform character of this losing stream allows a rough estimate (with ×3 uncertainty)
- **Hardhof operations**: Well-documented pumping and recharge volumes (±20%)
- **Western outflow**: Darcy-based estimate using groundwater gradients (±50%)

**What we cannot quantify with rough methods:**
- **Limmat natural infiltration**: The 2-order-of-magnitude uncertainty in leakage coefficients (3.5×10⁻⁷ to 3.5×10⁻⁵ s⁻¹), combined with spatially variable connectivity and the dominance of engineered Hardhof recharge, means any point estimate would be meaningless.

> **This is exactly why we need a calibrated groundwater model.**
>
> The perceptual model's purpose is not to achieve perfect closure, but to:
> 1. Identify the **dominant processes** (river-aquifer interaction dominates the budget)
> 2. Quantify what **can** be estimated with available data
> 3. Recognize what **cannot** be estimated and must be determined through calibration
> 4. Establish **initial parameter values** for the numerical model

---

### Comprehension Check

Before moving to Notebook 3, make sure you can answer:

> **Why can't we reliably estimate the Limmat's contribution to the water balance using the methods in this notebook?**

<details>
<summary>Click to check your answer</summary>

**Three reasons:**
1. The leakage coefficient spans **two orders of magnitude** (×100 range) along the river
2. The Limmat is **connected** to the groundwater table, so flux depends on both river stage AND aquifer head (unlike the simpler, disconnected Sihl)
3. The engineered **Hardhof system** dominates and overlaps with natural infiltration

This uncertainty is **why we need a calibrated numerical model** – it can constrain this parameter by matching simulated heads to observations (Step 5: Calibration).
</details>

## 7 — Could We Be Wrong? Alternative Conceptual Models

Our perceptual model makes several key assumptions: a single aquifer layer, steady-state conditions, and dominant river-aquifer interaction. But what if some of these assumptions are wrong? Professional guidelines stress that **considering alternative conceptual models** is essential — relying on a single interpretation is one of the greatest risks in groundwater modelling.

For the Limmat Valley, plausible alternatives include:

| Alternative | Different assumption | What would change |
|---|---|---|
| **Multi-layer system** | Aquitard separating shallow and deep flow | Vertical gradients, different K per layer |
| **Significant lateral inflow** | Hill recharge dominates over river leakage | Recharge boundaries become critical |
| **Transient recharge** | Seasonal or event-driven recharge pulses | Storage matters, steady-state invalid |

We proceed with our single-layer, river-dominated conceptual model because the available evidence (borehole logs, aquifer geometry, river proximity) supports it best. But we should revisit these alternatives if calibration reveals systematic misfits — for example, if residuals cluster near the valley margins, lateral inflow may be more important than assumed.

> **📚 Conceptual Surprise:**
> Bredehoeft (2005) coined the term "conceptual surprise" to describe cases where a well-calibrated model turns out to be based on the wrong conceptual understanding. A model can reproduce observed heads with the wrong processes if errors compensate each other — for example, overestimated recharge balanced by overestimated K. This is why calibration alone cannot validate a conceptual model; independent data types (flux measurements, tracers, geophysics) provide stronger tests. We revisit this idea in Notebooks 5 (calibration) and 6 (validation).

---

## What You're Taking Forward

From this perceptual model, you now have:

| For Notebook 3 (MODFLOW Fundamentals) | Value/Insight |
|---------------------------------------|---------------|
| Dominant process | River-aquifer interaction (especially Limmat) |
| Model area | ~10 km² |
| Initial K estimate | ~10⁻² m/s (864 m/day) for glaciofluvial gravels |
| Recharge estimate | ~10% of precipitation (~110 mm/year) |
| Key uncertainty | Limmat leakage coefficient (×100 range) |

**The big picture:** We know *what* fluxes matter, we have *rough estimates* for most, and we've identified *which parameter* needs calibration. The numerical model will formalize this understanding.

---

**Continue to:** [3_modflow_fundamentals.ipynb](3_modflow_fundamentals.ipynb) – Learn how MODFLOW 6 translates this perceptual model into numerical form

## References
[\[1\]](#1---The-Limmat-Valley-Aquifer) Hug J., and Beilick, A. (1934): Die Grundwasserverhältnisse des Kantons Zürich. In: Beiträge zur Geologie der Schweiz - Geotechnische Serie - Hydrologie. German. Available online here: https://scnat.ch/de/uuid/i/0bd7aa3b-0bd7-5d54-9e2f-597a42dada50-Die_Grundwasserverh%C3%A4ltnisse_des_Kantons_Z%C3%BCrich (accessed 2025-05-01)      
[\[2\]](#1---The-Limmat-Valley-Aquifer) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.    
[\[3\]](#1---The-Limmat-Valley-Aquifer) GIS-browser of the canton of Zurich: https://geo.zh.ch/ (accessed 2025-05-01)    
[\[4\]](#22---Top-Topography) Federal Office of Topography swisstopo, swissALTI3D height model (2m resolution), [https://www.swisstopo.admin.ch/en/height-model-swissalti3d](https://www.swisstopo.admin.ch/en/height-model-swissalti3d) (accessed 2025-05-01)    
[\[5\]](#23---Lateral-Model-Boundaries) Freeze, R. A., and Cherry, J. A. (1979): Groundwater. Prentice-Hall, Englewood Cliffs, New Jersey, USA. ISBN: 978-0133653122. Open-book available here: [https://gw-project.org/books/groundwater/](https://gw-project.org/books/groundwater/)   
[\[6\]](#3---Climate-Forcing) MeteoSwiss: https://www.meteoswiss.admin.ch/services-and-publications/applications/measurement-values.html#param=messnetz-klima&table=false&station=SMA&compare=y&chart=year (accessed 2025-05-01)  
[\[7\]](#3---Climate-Forcing) Opendata.swiss: Klimanormwerte, publisher: Federal Office of Meterology and Climatology. [https://opendata.swiss/en/dataset/klimanormwerte](https://opendata.swiss/en/dataset/klimanormwerte) (accessed 2025-05-01)    
[\[8\]](#3---Climate-Forcing) Lerner, David N. (2002): Identifying and quantifying urban recharge: a review. Hydrogeology Journal, Volume 10, Issue 1, pp 143–152. DOI: [https://doi.org/10.1007/s10040-001-0177-1](https://doi.org/10.1007/s10040-001-0177-1)    
[\[9\]](#3---Climate-Forcing) Qui, Guo Yu, Yan, Chunhua, Liu, Yuanbo (2023): Urban evapotranspiration and its effects on water budget and energy balance: Review and perspectives. Earth-Science Reviews, Volume 246. DOI: [https://doi.org/10.1016/j.earscirev.2023.104577](https://doi.org/10.1016/j.earscirev.2023.104577)       
[\[10\]](#4---River-aquifer-interaction) Locations of hydrological gauging stations maintained by the Federal Office for the Environment (FOEN): https://map.geo.admin.ch (accessed 2025-05-01)  
[\[11\]](#4---River-aquifer-interaction) Locations of hydrological gauging stations maintained by the cantonal office for waste, water, energy, and air (AWEL): https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/abfluss-wasserstand.html (accessed 2025-05-01)   
[\[12\]](#4---River-aquifer-interaction) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.   
[\[13\]](#5---Groundwater-pumping) City of Zurich, Drinking Water Supply: [https://www.stadt-zuerich.ch/de/umwelt-und-energie/wasser.html](https://www.stadt-zuerich.ch/de/umwelt-und-energie/wasser.html) (accessed 2025-08-21)    
[\[14\]](#5---Groundwater-pumping) Statistisches Jahrbuch der Stadt Zürich 2017 (Statistical yearbook of the city of Zurich 2017), Kapitel Wasser und Energie (Chapter water and energy), page 8: Wasserabgabe nach Wasserherkunft; Grundwasser (water deliveries by origin; groundwater).  [https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/publikationen-und-dienstleistungen/publikationen/jahrbuch.html](https://www.stadt-zuerich.ch/de/politik-und-verwaltung/statistik-und-daten/publikationen-und-dienstleistungen/publikationen/jahrbuch.html) (accessed 2025-08-21)      
[\[15\]](#6---Monitoring-the-Limmat-Valley-Aquifer) Locations of groundwater monitoring wells maintained by Federal Office for the Environment (FOEN): https://map.geo.admin.ch/ (accessed 2025-05-01)  
[\[16\]](#6---Monitoring-the-Limmat-Valley-Aquifer) Cantonal office of the environment (AWEL): https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/messdaten/grundwasserstaende.html (accessed 2025-05-01)
[\[17\]](#7---Could-We-Be-Wrong-Alternative-Conceptual-Models) Bredehoeft, J.D. (2005): The conceptualization model problem—surprise. Hydrogeology Journal, Volume 13, Issue 1, pp 37–46. DOI: [https://doi.org/10.1007/s10040-004-0430-5](https://doi.org/10.1007/s10040-004-0430-5)